In [1]:
# prepare_data.py

import pandas as pd
from sklearn.model_selection import train_test_split

print("--- Starting Data Preparation ---")

# --- CONFIG ---
# This should be your full, balanced dataset
INPUT_DATA_FILE = "data.csv" 
TEST_SET_SIZE = 0.15 # 15% for the final test set
VALIDATION_SET_SIZE = 0.1 # 10% of the remaining data for validation

# 1. Load the full dataset
try:
    df = pd.read_csv(INPUT_DATA_FILE, encoding='utf-8-sig')
    print(f"Loaded {len(df)} rows from '{INPUT_DATA_FILE}'.")
except FileNotFoundError:
    print(f"FATAL ERROR: Input data file not found at '{INPUT_DATA_FILE}'.")
    exit()

# 2. Create the harm_category for stratification
bins = [-0.1, 0.3, 0.6, 1.1]
labels = ['low', 'mid', 'high']
df['harm_category'] = pd.cut(df['harmscore'], bins=bins, labels=labels)

# 3. Split off the final, held-out test set
train_val_df, test_df = train_test_split(
    df, 
    test_size=TEST_SET_SIZE, 
    random_state=42, 
    stratify=df['harm_category']
)
print(f"Split off {len(test_df)} rows for the final test set.")

# 4. Split the remaining data into training and validation sets
train_df, val_df = train_test_split(
    train_val_df,
    test_size=VALIDATION_SET_SIZE,
    random_state=42,
    stratify=train_val_df['harm_category']
)
print(f"Remaining data split into {len(train_df)} training rows and {len(val_df)} validation rows.")

# 5. Save the three distinct datasets
train_df.to_csv('train_data.csv', index=False)
val_df.to_csv('val_data.csv', index=False)
test_df.to_csv('final_test_set.csv', index=False)

print("\n--- Data Preparation Complete ---")
print("Successfully created:")
print(f"  - train_data.csv ({len(train_df)} rows)")
print(f"  - val_data.csv ({len(val_df)} rows)")
print(f"  - final_test_set.csv ({len(test_df)} rows)")

--- Starting Data Preparation ---
Loaded 6115 rows from 'data.csv'.
Split off 918 rows for the final test set.
Remaining data split into 4677 training rows and 520 validation rows.

--- Data Preparation Complete ---
Successfully created:
  - train_data.csv (4677 rows)
  - val_data.csv (520 rows)
  - final_test_set.csv (918 rows)


In [2]:
import torch
from torch.utils.data import DataLoader
from transformers import RobertaTokenizer
from transformers import get_linear_schedule_with_warmup
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import spearmanr, pearsonr
from tqdm import tqdm

from harm_model import HarmScoringModel
from utils import PromptDataset

# Config
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS = 4
BATCH_SIZE = 12
LR = 1e-5

# Load pre-split data
print("Loading pre-split training and validation data...")
train_df = pd.read_csv('train_data.csv', encoding='utf-8-sig')
val_df = pd.read_csv('val_data.csv', encoding='utf-8-sig')


# Tokenizer and datasets
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
train_dataset = PromptDataset(train_df, tokenizer)
val_dataset = PromptDataset(val_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

# Model, optimizer, loss
model = HarmScoringModel().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
loss_fn = torch.nn.BCEWithLogitsLoss()

# Calculate total training steps
total_steps = len(train_loader) * EPOCHS

# Create the scheduler
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0, # Default, no warmup
    num_training_steps=total_steps
)

def evaluate(model, val_loader, epoch):
    model.eval()
    total_loss = 0

    all_labels = []
    all_preds = []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['label'].to(DEVICE)

            outputs = model(input_ids, attention_mask)
            loss = loss_fn(outputs, labels)
            total_loss += loss.item()

            preds = torch.sigmoid(outputs)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)

    # ---------------- GLOBAL REGRESSION METRICS ----------------
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    spearman = spearmanr(y_true, y_pred).correlation
    pearson = pearsonr(y_true, y_pred)[0]

    print(f"\n--- Regression Metrics (Epoch {epoch+1}) ---")
    print(f"MAE      : {mae:.4f}")
    print(f"RMSE     : {rmse:.4f}")
    print(f"R²       : {r2:.4f}")
    print(f"Spearman : {spearman:.4f}")
    print(f"Pearson  : {pearson:.4f}")

    # ---------------- PER-BIN ERROR ANALYSIS ----------------
    bins = [-0.1, 0.3, 0.6, 1.1]
    bin_names = ['low', 'mid', 'high']

    df = pd.DataFrame({
        "y_true": y_true,
        "y_pred": y_pred,
        "bin": pd.cut(y_true, bins=bins, labels=bin_names)
    })

    df["abs_error"] = np.abs(df.y_true - df.y_pred)
    df["signed_error"] = df.y_pred - df.y_true

    print("\n--- Error by Harm Level (Ground Truth Bins) ---")
    print(
        df.groupby("bin").agg(
            count=("y_true", "count"),
            MAE=("abs_error", "mean"),
            RMSE=("abs_error", lambda x: np.sqrt(np.mean(x ** 2))),
            Mean_True=("y_true", "mean"),
            Mean_Pred=("y_pred", "mean"),
            Bias=("signed_error", "mean")
        )
    )

    # ---------------- HIGH-HARM FAILURE DIAGNOSTIC ----------------
    high_df = df[df["bin"] == "high"]
    if len(high_df) > 0:
        under_rate = np.mean(high_df.y_pred < 0.6)
        mean_under = np.mean(high_df.y_true - high_df.y_pred)
        print("\nHigh-harm underestimation rate:", round(under_rate, 3))
        print("Mean underestimation magnitude:", round(mean_under, 4))

    print("--------------------------------------------------\n")

    return total_loss / len(val_loader)


# Training loop
for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0
    for batch in tqdm(train_loader):
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['label'].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_train_loss += loss.item()

    val_loss = evaluate(model, val_loader, epoch)
    print(f"Epoch {epoch+1} | Total train loss: {total_train_loss}")
    print(f"Epoch {epoch+1} | Train Loss: {total_train_loss / len(train_loader):.4f} | Val Loss: {val_loss:.4f}")

torch.save(model.state_dict(), 'baseline_model.pt')


g:\T2430392\Efficient-Distilled-miniLLM-to-Prevent-Jailbreak-Attacks\jailbreak\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading pre-split training and validation data...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 390/390 [00:21<00:00, 18.43it/s]
C:\Users\T2430392\AppData\Local\Temp\ipykernel_51308\4290328897.py:103: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("bin").agg(



--- Regression Metrics (Epoch 1) ---
MAE      : 0.0510
RMSE     : 0.0727
R²       : 0.9471
Spearman : 0.9348
Pearson  : 0.9756

--- Error by Harm Level (Ground Truth Bins) ---
      count       MAE      RMSE  Mean_True  Mean_Pred      Bias
bin                                                            
low     177  0.033514  0.043791   0.111582   0.099859 -0.011723
mid     164  0.048796  0.065715   0.447561   0.405154 -0.042407
high    179  0.070277  0.097488   0.850536   0.837543 -0.012993

High-harm underestimation rate: 0.0
Mean underestimation magnitude: 0.013
--------------------------------------------------

Epoch 1 | Total train loss: 189.65721541643143
Epoch 1 | Train Loss: 0.4863 | Val Loss: 0.4692


100%|██████████| 390/390 [00:17<00:00, 22.43it/s]
C:\Users\T2430392\AppData\Local\Temp\ipykernel_51308\4290328897.py:103: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("bin").agg(


...

100%|██████████| 390/390 [00:17<00:00, 21.98it/s]
C:\Users\T2430392\AppData\Local\Temp\ipykernel_51308\4290328897.py:103: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("bin").agg(



--- Regression Metrics (Epoch 3) ---
MAE      : 0.0469
RMSE     : 0.0696
R²       : 0.9515
Spearman : 0.9376
Pearson  : 0.9764

--- Error by Harm Level (Ground Truth Bins) ---
      count       MAE      RMSE  Mean_True  Mean_Pred      Bias
bin                                                            
low     177  0.027259  0.037080   0.111582   0.109003 -0.002579
mid     164  0.049607  0.050165   0.447561   0.449520  0.001959
high    179  0.063890  0.102028   0.850536   0.872697  0.022162

High-harm underestimation rate: 0.011
Mean underestimation magnitude: -0.0222
--------------------------------------------------

Epoch 3 | Total train loss: 180.9669238626957
Epoch 3 | Train Loss: 0.4640 | Val Loss: 0.4686


100%|██████████| 390/390 [00:17<00:00, 21.89it/s]



--- Regression Metrics (Epoch 4) ---
MAE      : 0.0467
RMSE     : 0.0689
R²       : 0.9525
Spearman : 0.9385
Pearson  : 0.9761

--- Error by Harm Level (Ground Truth Bins) ---
      count       MAE      RMSE  Mean_True  Mean_Pred      Bias
bin                                                            
low     177  0.026256  0.037015   0.111582   0.109109 -0.002473
mid     164  0.049551  0.049996   0.447561   0.449614  0.002053
high    179  0.064249  0.100688   0.850536   0.856029  0.005494

High-harm underestimation rate: 0.011
Mean underestimation magnitude: -0.0055
--------------------------------------------------

Epoch 4 | Total train loss: 180.43216735124588
Epoch 4 | Train Loss: 0.4626 | Val Loss: 0.4677


C:\Users\T2430392\AppData\Local\Temp\ipykernel_51308\4290328897.py:103: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("bin").agg(


In [8]:
import torch
import torch.nn as nn
import pandas as pd
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import spearmanr, pearsonr


# --- Import your custom model and dataset classes ---
from harm_model import HarmScoringModel
from utils import PromptDataset

# ==============================================================================
# --- CONFIG: UPDATE THESE AS NEEDED ---
# ==============================================================================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- File Paths ---
JIGSAW_DATA_PATH = "jigsaw.csv"
#HARMSCORE_DATA_PATH = "data.csv" 
INTERMEDIATE_MODEL_PATH = "roberta-toxic.pt"
FINAL_MODEL_PATH = "transfer_learning_model.pt"

# --- Stage 1: Jigsaw Training Hyperparameters ---
EPOCHS_JIGSAW = 1
LR_JIGSAW = 2e-5
BATCH_SIZE_JIGSAW = 16 

# --- Stage 2: Harmscore Training Hyperparameters (Matching your train.py) ---
EPOCHS_FINAL = 3
LR_FINAL = 1e-5
BATCH_SIZE_FINAL = 12

# --- General Config ---
TOKENIZER_NAME = "roberta-base"
MAX_LENGTH = 128
WEIGHT_DECAY = 0.01
# ==============================================================================


# ==============================================================================
# --- STAGE 1: JIGSAW DATASET AND CLASSIFICATION MODEL ---
# ==============================================================================

class JigsawDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.tokenizer = tokenizer
        self.texts = dataframe['comment_text'].tolist()
        self.labels = dataframe[['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']].values
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.float)
        }

class ToxicityClassifierModel(nn.Module):
    def __init__(self, model_name='roberta-base'):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained(model_name)
        self.classifier = nn.Linear(self.roberta.config.hidden_size, 6)

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        logits = self.classifier(pooled_output)
        return logits

# ==============================================================================
# --- STAGE 2: EVALUATION FUNCTION (ADAPTED FROM YOUR train.py) ---
# ==============================================================================

def evaluate_final(model, val_loader, loss_fn, epoch):
    model.eval()
    total_loss = 0

    all_labels = []
    all_preds = []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['label'].to(DEVICE)

            outputs = model(input_ids, attention_mask)
            loss = loss_fn(outputs, labels)
            total_loss += loss.item()

            preds = torch.sigmoid(outputs)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    spearman = spearmanr(y_true, y_pred).correlation
    pearson = pearsonr(y_true, y_pred)[0]

    print(f"\n--- Regression Metrics (Epoch {epoch+1}) ---")
    print(f"MAE      : {mae:.4f}")
    print(f"RMSE     : {rmse:.4f}")
    print(f"R²       : {r2:.4f}")
    print(f"Spearman : {spearman:.4f}")
    print(f"Pearson  : {pearson:.4f}")

    bins = [-0.1, 0.3, 0.6, 1.1]
    bin_names = ['low', 'mid', 'high']

    df = pd.DataFrame({
        "y_true": y_true,
        "y_pred": y_pred,
        "bin": pd.cut(y_true, bins=bins, labels=bin_names)
    })

    df["abs_error"] = np.abs(df.y_true - df.y_pred)
    df["signed_error"] = df.y_pred - df.y_true

    print("\n--- Error by Harm Level (Ground Truth Bins) ---")
    print(
        df.groupby("bin").agg(
            count=("y_true", "count"),
            MAE=("abs_error", "mean"),
            RMSE=("abs_error", lambda x: np.sqrt(np.mean(x ** 2))),
            Mean_True=("y_true", "mean"),
            Mean_Pred=("y_pred", "mean"),
            Bias=("signed_error", "mean")
        )
    )

    high_df = df[df["bin"] == "high"]
    if len(high_df) > 0:
        under_rate = np.mean(high_df.y_pred < 0.6)
        mean_under = np.mean(high_df.y_true - high_df.y_pred)
        print("\nHigh-harm underestimation rate:", round(under_rate, 3))
        print("Mean underestimation magnitude:", round(mean_under, 4))

    print("--------------------------------------------------\n")

    return total_loss / len(val_loader)

# ==============================================================================
# --- MAIN EXECUTION SCRIPT ---
# ==============================================================================

if __name__ == "__main__":
    print(f"Using device: {DEVICE}")
    tokenizer = RobertaTokenizer.from_pretrained(TOKENIZER_NAME)

    # --------------------------------------------------------------------------
    # STAGE 1: Intermediate Training on Jigsaw
    # --------------------------------------------------------------------------
    print("\n" + "="*50)
    print("      STAGE 1: INTERMEDIATE TRAINING ON JIGSAW")
    print("="*50 + "\n")

    try:
        jigsaw_df = pd.read_csv(JIGSAW_DATA_PATH)
    except FileNotFoundError:
        print(f"FATAL ERROR: Jigsaw data not found at '{JIGSAW_DATA_PATH}'.")
        exit()

    jigsaw_dataset = JigsawDataset(jigsaw_df, tokenizer, max_length=MAX_LENGTH)
    jigsaw_loader = DataLoader(jigsaw_dataset, batch_size=BATCH_SIZE_JIGSAW, shuffle=True)
    
    model_jigsaw = ToxicityClassifierModel(model_name=TOKENIZER_NAME).to(DEVICE)
    optimizer = AdamW(model_jigsaw.parameters(), lr=LR_JIGSAW, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.MSELoss()
    
    total_steps_jigsaw = len(jigsaw_loader) * EPOCHS_JIGSAW
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=0, num_training_steps=total_steps_jigsaw
    )

    print(f"Starting Jigsaw training for {EPOCHS_JIGSAW} epoch(s)...")
    for epoch in range(EPOCHS_JIGSAW):
        model_jigsaw.train()
        loop = tqdm(jigsaw_loader, leave=True)
        for batch in loop:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            
            outputs = model_jigsaw(input_ids, attention_mask)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
            
            loop.set_description(f"Epoch {epoch+1}")
            loop.set_postfix(loss=loss.item())

    print(f"Intermediate training complete. Saving model to '{INTERMEDIATE_MODEL_PATH}'...")
    torch.save(model_jigsaw.state_dict(), INTERMEDIATE_MODEL_PATH)
    del model_jigsaw, jigsaw_df, jigsaw_dataset, jigsaw_loader, optimizer, scheduler
    torch.cuda.empty_cache()

    # --------------------------------------------------------------------------
    # STAGE 2: Final Fine-tuning on Harmscore Data
    # --------------------------------------------------------------------------
    print("\n" + "="*50)
    print("      STAGE 2: FINAL FINE-TUNING ON HARMSCORE DATA")
    print("="*50 + "\n")


    # Load pre-split data
    print("Loading pre-split training and validation data...")
    train_df = pd.read_csv('train_data.csv', encoding='utf-8-sig')
    val_df = pd.read_csv('val_data.csv', encoding='utf-8-sig')
    
    train_dataset = PromptDataset(train_df, tokenizer, max_length=MAX_LENGTH)
    val_dataset = PromptDataset(val_df, tokenizer, max_length=MAX_LENGTH)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE_FINAL, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE_FINAL)

    final_model = HarmScoringModel(model_name=TOKENIZER_NAME).to(DEVICE)

    print(f"\nLoading weights from intermediate model '{INTERMEDIATE_MODEL_PATH}'...")
    intermediate_weights = torch.load(INTERMEDIATE_MODEL_PATH, map_location=DEVICE)
    final_model.load_state_dict(intermediate_weights, strict=False)
    print("Successfully loaded RoBERTa weights. The regression head is randomly initialized.")

    optimizer = AdamW(final_model.parameters(), lr=LR_FINAL, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.BCEWithLogitsLoss()
    total_steps_final = len(train_loader) * EPOCHS_FINAL
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=0, num_training_steps=total_steps_final
    )

    print(f"\nStarting final fine-tuning for {EPOCHS_FINAL} epoch(s)...")
    for epoch in range(EPOCHS_FINAL):
        final_model.train()
        total_train_loss = 0
        loop = tqdm(train_loader, leave=True)
        for batch in loop:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['label'].to(DEVICE)

            outputs = final_model(input_ids, attention_mask)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
            
            total_train_loss += loss.item()
            loop.set_description(f"Epoch {epoch+1}")
            loop.set_postfix(train_loss=loss.item())
        
        avg_train_loss = total_train_loss / len(train_loader)
        
        # Evaluate at the end of each epoch
        val_loss = evaluate_final(final_model, val_loader, loss_fn, epoch)
        print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f}")

    print(f"\nFinal training complete. Saving best model to '{FINAL_MODEL_PATH}'...")
    torch.save(final_model.state_dict(), FINAL_MODEL_PATH)
    print("Process finished successfully.")

Using device: cuda

      STAGE 1: INTERMEDIATE TRAINING ON JIGSAW



Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Starting Jigsaw training for 1 epoch(s)...


Epoch 1: 100%|██████████| 9974/9974 [09:29<00:00, 17.51it/s, loss=3.69e-5] 


Intermediate training complete. Saving model to 'roberta-toxic.pt'...

      STAGE 2: FINAL FINE-TUNING ON HARMSCORE DATA

Loading pre-split training and validation data...


...


--- Regression Metrics (Epoch 2) ---
MAE      : 0.0487
RMSE     : 0.0673
R²       : 0.9547
Spearman : 0.9360
Pearson  : 0.9779

--- Error by Harm Level (Ground Truth Bins) ---
      count       MAE      RMSE  Mean_True  Mean_Pred      Bias
bin                                                            
low     177  0.032097  0.043393   0.111582   0.094385 -0.017197
mid     164  0.049101  0.052927   0.447561   0.434825 -0.012736
high    179  0.064776  0.093360   0.850536   0.843903 -0.006632

High-harm underestimation rate: 0.0
Mean underestimation magnitude: 0.0066
--------------------------------------------------

Epoch 2 | Train Loss: 0.4671 | Val Loss: 0.4673


Epoch 3: 100%|██████████| 390/390 [00:18<00:00, 21.50it/s, train_loss=0.465]



--- Regression Metrics (Epoch 3) ---
MAE      : 0.0469
RMSE     : 0.0656
R²       : 0.9569
Spearman : 0.9368
Pearson  : 0.9783

--- Error by Harm Level (Ground Truth Bins) ---
      count       MAE      RMSE  Mean_True  Mean_Pred      Bias
bin                                                            
low     177  0.029065  0.037894   0.111582   0.109456 -0.002126
mid     164  0.049998  0.050582   0.447561   0.452925  0.005364
high    179  0.061821  0.093512   0.850536   0.855972  0.005437

High-harm underestimation rate: 0.0
Mean underestimation magnitude: -0.0054
--------------------------------------------------

Epoch 3 | Train Loss: 0.4656 | Val Loss: 0.4666

Final training complete. Saving best model to 'transfer_learning_model.pt'...


C:\Users\T2430392\AppData\Local\Temp\ipykernel_51308\4162130319.py:140: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("bin").agg(


Process finished successfully.
